**Українська версія:**      
Щоб побудувати та визначити модель, яка забезпечує найнадійніші прогнози відтоку клієнтів, буде навчено, оцінено та збережено п'ять моделей: **логістичну регресію, дерево рішень, випадковий ліс, метод опорних векторів (SVM) та нейронну мережу**. Моделі представлені у порядку зростання складності — починаючи з логістичної регресії як інтерпретованої базової моделі, і завершуючи нейронною мережею. Усі п'ять моделей буде порівняно за однаковим набором метрик (Accuracy, Precision, Recall, F1-score), після чого модель з найкращими результатами буде обрана як фінальна — проте інші моделі також залишаться прийнятними варіантами залежно від конкретного балансу між інтерпретованістю, швидкістю та точністю, який потрібен у певній ситуації.

**English version:**       
In order to build and identify the model that delivers the most reliable churn predictions, five models will be trained, evaluated, and saved: **Logistic Regression, Decision Tree, Random Forest, Support Vector Machine, and a Neural Network**. The models are presented in increasing order of complexity, starting with Logistic Regression as an interpretable baseline and finishing with a Neural Network. All five will be compared using the same set of metrics (Accuracy, Precision, Recall, F1-score), after which the best-performing model will be selected as the final choice — though the remaining models will also remain valid options depending on the specific trade-off between interpretability, speed, and accuracy required. 

In [1]:
import os
import joblib

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, classification_report, hinge_loss, ConfusionMatrixDisplay, RocCurveDisplay

In [2]:
df = pd.read_csv('../data/processed/df_cleaned.csv')
df

,is_tv_subscriber,is_movie_package_subscriber,subscription_age,bill_avg,remaining_contract,service_failure_count,download_avg,upload_avg,download_over_limit,churn,has_contract_info,has_active_contract
0,1,0,11.95,25,0.14,0,8.4,2.3,0,0,1,1
1,0,0,8.22,0,0.00,0,0.0,0.0,0,1,0,0
2,1,0,8.91,16,0.00,0,13.7,0.9,0,1,1,0
3,0,0,6.87,21,0.00,1,0.0,0.0,0,1,0,0
4,0,0,6.39,0,0.00,0,0.0,0.0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
72269,1,1,0.09,0,1.25,0,0.0,0.0,0,1,1,1
72270,1,0,0.06,1,1.63,0,0.8,0.0,0,1,1,1
72271,1,0,0.02,0,2.19,0,1.5,0.2,0,1,1,1
72272,0,0,0.01,0,0.72,0,0.0,0.0,0,1,1,1


In [3]:
X = df.drop(columns=['churn'])
y = df['churn']

In [4]:
X_train, X_test,y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=19)

**Українська версія:**     
Усі п'ять моделей навчаються на одному й тому самому стандартизованому наборі ознак, а не з вибірковим масштабуванням для кожної моделі окремо. Хоча стандартизація не є строго обов'язковою для деревоподібних моделей (дерево рішень, випадковий ліс), оскільки їхні розділення базуються на порогових порівняннях і тому не залежать від масштабу ознак, уніфіковане застосування спрощує пайплайн і усуває ризик випадково подати не той варіант даних не тій моделі. Для SVM та нейронної мережі стандартизація є необхідною, оскільки ці моделі за своєю природою чутливі до масштабу ознак. Для логістичної регресії масштабування не є математично обов'язковим для нерегуляризованої моделі, проте є усталеною доброю практикою: воно забезпечує справедливе застосування L2-регуляризації незалежно від початкового масштабу ознак і покращує збіжність оптимізатора — обидва фактори зазвичай призводять до кращої, точнішої моделі на практиці.

**English version:**    
All five models are trained on the same standardized feature set, rather than applying scaling selectively per model. While standardization is not strictly required for tree-based models (Decision Tree, Random Forest), since their splits are based on threshold comparisons and are therefore scale-invariant, applying it uniformly simplifies the pipeline and eliminates the risk of accidentally feeding the wrong data version to the wrong model. For SVM and the Neural Network, standardization is essential, as these models are inherently sensitive to feature scale. For Logistic Regression, scaling is not mathematically required for an unregularized model, but it is standard best practice: it ensures L2 regularization penalizes features fairly regardless of their original scale, and it improves solver convergence — both of which typically lead to a better-fitted, more accurate model in practice.

In [5]:
# Українська версія: Стандартизація оброблюваних даних
# English version: Standartization of processed data

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 1. Модель логістичної регресії (Logistic Regression Model)

In [6]:
num_cycle = 0

In [8]:
dict_all_best_params_per_step = {}
all_metrics_per_step = {}

In [43]:
# Українська версія: Визначення типу моделі, параметрів та класифікатора.
# English version: Definition of the model type, parameters and classifier. 
model_lg = LogisticRegression(solver='saga',max_iter=5000,random_state=19)


parameters = [
    {
        'C': [5,7,7.2,7.5,7.6],
        'penalty': ['l1', 'l2'],
        'class_weight':['balanced', None]     
    },
]

clf = GridSearchCV(estimator=model_lg,param_grid=parameters)
num_cycle += 1

**Українська версія:**
Щоб знайти оптимальне значення C, буде протестовано кілька значень навколо поточного найкращого параметра (C=10). Усі попередньо протестовані комбінації параметрів зберігаються у `dict_all_best_params_per_step`, щоб уникнути повторного проведення експериментів.

**English version:**  
To find the optimal value of C, several values around the temporary best parameter will be tested. The previously tested parameter combinations are saved in `dict_all_best_params_per_step` to avoid repeating experiments.

In [44]:
# Train model
clf.fit(X_train_scaled, y_train)

# Predictions
y_pred_lg = clf.predict(X_test_scaled)

# Accuracy score
acc_lg = accuracy_score(y_pred_lg, y_test)

# Loss score
y_pred_proba_lg = clf.predict_proba(X_test_scaled)[:,1]
loss_lg = log_loss(y_test,y_pred_proba_lg)

In [66]:
class_rep_lr_dict = classification_report(y_test, y_pred_lg,output_dict=True)
class_rep_lr_str = classification_report(y_test, y_pred_lg,output_dict=False)


print('Показник точності логістичної регресії: ', acc_lg)
print('Показник втрат у логістичній регресії: ', loss_lg)
print('\t\tПідсумкова оцінка моделі:')
# print('Accuracy score of Logistic Regression: ', acc_lg)
# print('Loss score of Logistic Regression: ', loss_lg)

print(class_rep_lr_str)

Показник точності логістичної регресії:  0.9236250432376341
Показник втрат у логістичній регресії:  0.23644608009457616
		Підсумкова оцінка моделі:
              precision    recall  f1-score   support

           0       0.89      0.94      0.92      6445
           1       0.95      0.91      0.93      8010

    accuracy                           0.92     14455
   macro avg       0.92      0.93      0.92     14455
weighted avg       0.93      0.92      0.92     14455



In [67]:

precision_churn = class_rep_lr['1']['precision']
recall_churn = class_rep_lr['1']['recall']
f1_churn = class_rep_lr['1']['f1-score']

final_comparison_churn_df = pd.DataFrame({'Metrics':['Accuracy', 'Loss','Precision','Recall','F1-Score'],
                                          'best_logistic_regression_model': [acc_lg,loss_lg,precision_churn,recall_churn,
                                                                             f1_churn]})

precision_no_churn = class_rep_lr['0']['precision']
recall_no_churn = class_rep_lr['0']['recall']
f1_no_churn = class_rep_lr['0']['f1-score']

final_comparison_no_churn_df = pd.DataFrame({'Metrics':['Accuracy', 'Loss','Precision','Recall','F1-Score'],
                                          'best_logistic_regression_model': [acc_lg,loss_lg,precision_no_churn,recall_no_churn,
                                                                             f1_no_churn]})

macro_precision_lg = class_rep_lr['macro avg']['precision']
macro_recall_lg = class_rep_lr['macro avg']['recall']
macro_f1_lg = class_rep_lr['macro avg']['f1-score']

final_comparison_macro_df = pd.DataFrame({'Metrics':['Accuracy', 'Loss','Precision','Recall','F1-Score'],
                                          'best_logistic_regression_model': [acc_lg,loss_lg,macro_precision_lg,macro_recall_lg,
                                                                             macro_f1_lg]})
final_comparison_macro_df

,Metrics,best_logistic_regression_model
0,Accuracy,0.923625
1,Loss,0.236446
2,Precision,0.921917
3,Recall,0.925584
4,F1-Score,0.923107


In [47]:
best_params = clf.best_params_
best_estimator = clf.best_estimator_
dict_all_best_params_per_step[f'{num_cycle}_cycle'] = best_params
all_metrics_per_step[f'{num_cycle}_cycle'] = class_rep_lr
print('Найкращі параметри: ', best_params)
print('Найкраща логістична модель: ', best_estimator)
# print('Best parameters are: ', best_params)
# print('Best logistic model is: ', best_estimator)

Найкращі параметри:  {'C': 7.5, 'class_weight': None, 'penalty': 'l2'}
Найкраща логістична модель:  LogisticRegression(C=7.5, max_iter=5000, random_state=19, solver='saga')


In [99]:
## Українська версія: Збереження моделі у папці logistic_regression
## English version: Saving the model to the logistic_regression folder
logistic_models_location = '../models/logistic_regression'

os.makedirs(logistic_models_location, exist_ok=True)

model_path = os.path.join(logistic_models_location, 'best_logistic_regression_model.pkl')

joblib.dump(best_estimator,model_path)

['../models/logistic_regression/best_logistic_regression_model.pkl']

### 2. Модель «дерево рішень» (The Decision Tree Model)

In [104]:
# 1. Initialize the model
decision_tree_model = DecisionTreeClassifier(random_state=19)

# 2. Parameters definition
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 2, 4]
}
# 3. Classifier 
clf_dt = GridSearchCV(estimator=decision_tree_model,param_grid=param_grid,cv=10)

# 4. Training
clf_dt.fit(X_train_scaled, y_train)

# 5. Prediction
y_pred_dt = clf_dt.predict(X_test_scaled)

# 6. Calculate accuracy and other metrics
acc_dt = accuracy_score(y_test, y_pred_dt)
class_report_dt = classification_report(y_test, y_pred_dt)

In [106]:
acc_dt_rounded = round(acc_dt * 100, 2)
print(f'The accuracy score of the Decision Tree model is {acc_dt_rounded} ({acc_dt})')

print('\t\t')
print('\t\tПідсумкова оцінка моделі:')
print(class_report_dt)

The accuracy score of the Decision Tree model is 93.91 (0.9390522310619163)
		Classification report for Decision Tree model :
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      6445
           1       0.96      0.93      0.94      8010

    accuracy                           0.94     14455
   macro avg       0.94      0.94      0.94     14455
weighted avg       0.94      0.94      0.94     14455



In [108]:
best_params_dt = clf_dt.best_params_
best_estimator_dt = clf_dt.best_estimator_
print('Best parameters are: ', best_params_dt)
print('Best logistic model is: ', best_estimator_dt)

Best parameters are:  {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 4}
Best logistic model is:  DecisionTreeClassifier(criterion='entropy', max_depth=10, min_samples_leaf=4,
                       random_state=19)


If time allows, build a visual graphic

In [111]:
decision_tree_location = '../models/decision_tree'

os.makedirs(decision_tree_location, exist_ok=True)

model_path = os.path.join(decision_tree_location, 'best_decision_tree_model.pkl')

joblib.dump(best_estimator_dt,model_path)

['../models/decision_tree/best_decision_tree_model.pkl']

### 3. Модель «Випадковий ліс» (The Random Forest Model)

In [113]:
# 1. Initialize the model
random_forest_model = RandomForestClassifier(random_state=19)

# 2. Parameters definition
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 2, 4]
}
# 3. Classifier 
clf_rf = GridSearchCV(estimator=random_forest_model,param_grid=param_grid,cv=10)

# 4. Training
clf_rf.fit(X_train_scaled, y_train)

# 5. Prediction
y_pred_rf = clf_rf.predict(X_test_scaled)

# 6. Calculate accuracy and other metrics
acc_rf = accuracy_score(y_test, y_pred_rf)
class_report_rf = classification_report(y_test, y_pred_rf)

In [117]:
acc_rf_rounded = round(acc_rf * 100, 2)
print(f'The accuracy score of the Random Forest Model is {acc_rf_rounded} ({acc_rf})')
# print('Loss score of Logistic Regression: ', loss_lg)
print('\t\tClassification report for the Random Forest model:')
print(class_report_rf)

The accuracy score of the Random Forest Model is 94.1 (0.940989277066759)
		Classification report for the Random Forest model:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      6445
           1       0.96      0.94      0.95      8010

    accuracy                           0.94     14455
   macro avg       0.94      0.94      0.94     14455
weighted avg       0.94      0.94      0.94     14455



### 4. Модель «Метод опорних векторів» (Support Vector Machine Model / SVM)

In [ ]:
##  {'kernel': ['poly'],'C': [0.01, 0.1, 1, 10, 100],'degree': [2, 3, 4], 'gamma': ['scale']}
##
# 1. Initialize the model
svm_model = SVC(random_state=19)

# 2. Parameters definition
parameters_svw = [
    {
        'kernel': ['linear'],
        'C': [0.01, 0.1, 1, 10, 100] 
    },
    {
        'kernel': ['rbf'],
        'C': [0.01, 0.1, 1, 10, 100],
        'gamma': ['scale']
    }
]
# 3. Classifier 
clf_svm = GridSearchCV(estimator=svm_model,param_grid=parameters_svw,cv=10)

# 4. Training
clf_svm.fit(X_train_scaled, y_train)

# 5. Prediction
y_pred_svm = clf_svm.predict(X_test_scaled)

# 6. Calculate accuracy and other metrics
acc_svm = accuracy_score(y_test, y_pred_svm)
class_report_svm = classification_report(y_test, y_pred_svm)

In [ ]:
acc_svm_rounded = round(acc_svm * 100, 2)
print(f'The accuracy score of the Random Forest Model is {acc_svm_rounded} ({acc_svm})')
# print('Loss score of Logistic Regression: ', loss_lg)
print('\t\tClassification report for the Random Forest model:')
print(class_report_rf)